# GRM shard timing / feasibility check

Before committing to a full sharded GRM run (`plink --make-grm-bin --parallel k n`
across `n` shards), this times a handful of representative shards for a
candidate `n` (200, per the initial sizing conversation), checks whether
PLINK's row-balancing actually keeps per-shard cost uniform across `k`
(theory says it should -- the split balances off-diagonal *pair* counts, not
raw row counts -- but this repo's convention throughout is to measure, not
assume: `chr22_qc_thinning_timing.ipynb` timed one chromosome before the
genome-wide QC run, `THIN_P` was calibrated empirically), and extrapolates
total single-threaded compute plus a feasibility table of wall-clock
estimates at a few concurrency levels.

**Not a production run.** This doesn't compute the full GRM -- it's the same
role `chr22_qc_thinning_timing.ipynb` played for QC: a calibration step whose
numbers feed into the real sharded-GRM notebook (not yet built) once a shard
count is actually decided.

## Compute resource

Same `n1-highmem-64` (64 vCPU / 416 GB) as the QC + merge step. The
concurrency bound here is different, though: PLINK 1.9's `--make-grm-bin`
is single-threaded (no useful `--threads` lever), and every shard -- however
few rows it covers -- still loads the **entire** packed genotype panel into
memory, since any row's relatedness needs every other sample's genotypes
too. So concurrency is bounded by RAM ÷ (per-shard memory), not vCPU count.
This notebook measures actual peak memory per shard (`/usr/bin/time -v`)
rather than estimating it from the packed file size, since plink's real
working-set can be a multiple of the raw packed data.

## Setup

PLINK 1.9, not plink2 -- `GRM-pairs/grm_bin_sharded`'s row-range recovery
(used later, for the phenotype cross-product step) is calibrated to PLINK
1.9's own `--parallel` split algorithm specifically. This notebook is plink
only -- no `GRM-pairs` build or code path touched here.

In [ ]:
%%bash
set -e

BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink" ]; then
  # URL is dated; if it 404s, get current link from https://www.cog-genomics.org/plink/1.9/
  PLINK_URL="https://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20230116.zip"
  cd /tmp
  wget -q -O plink.zip "$PLINK_URL"
  unzip -o -q plink.zip plink -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink"
fi

export PATH="$BIN_DIR:$PATH"
plink --version

# GNU time (/usr/bin/time -v, for peak memory below) -- distinct from bash's own
# built-in `time` keyword, which doesn't support -v, and not guaranteed preinstalled
if ! command -v /usr/bin/time >/dev/null 2>&1; then
  sudo apt-get update -qq && sudo apt-get install -y -qq time
fi
/usr/bin/time --version

nproc
free -h

In [ ]:
import os

bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"

## Paths

`BED_PREFIX`: the genome-wide PLINK1 bed/bim/fam `genome_wide_qc_thinning_merge.ipynb`
persisted to the bucket. Copied to local scratch first, same convention as
everywhere else in this pipeline -- plink reading a multi-GB bed repeatedly
over the gcsfuse-mounted bucket is much slower than local disk.

In [ ]:
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
BUCKET_DIR = f"{WORKSPACE_BUCKET}/data/03_grm_shards"

LOCAL_WORK_DIR = os.path.expanduser("~/scratch_grm")
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

BED_NAME = "genome_wide_round2b_thinned_bed"   # from genome_wide_qc_thinning_merge.ipynb
BED_PREFIX = os.path.join(LOCAL_WORK_DIR, BED_NAME)

for ext in ("bed", "bim", "fam"):
    bucket_path = f"{BUCKET_DIR}/{BED_NAME}.{ext}"
    local_path = f"{BED_PREFIX}.{ext}"
    assert os.path.isfile(bucket_path), f"missing merged bed export: {bucket_path!r} -- run genome_wide_qc_thinning_merge.ipynb's merge section first"
    if not os.path.isfile(local_path):
        import shutil
        shutil.copy(bucket_path, local_path)

print(BED_PREFIX)

In [ ]:
%%bash -s "$BED_PREFIX"
set -e
BED_PREFIX=$1

echo "N (samples):"
wc -l < "${BED_PREFIX}.fam"
echo "M (variants):"
wc -l < "${BED_PREFIX}.bim"
echo "bed file size:"
ls -lh "${BED_PREFIX}.bed" | awk '{print $5}'

## Run parameters

`N_SHARDS = 200`: the candidate to check, not yet a decision. `SAMPLE_KS`:
a handful of shards spread across the full range (first, ~25%, ~50%, ~75%,
last), not just one -- if PLINK's row-balancing is working as documented,
these should all take roughly the same time despite covering different row
ranges; if they don't, that's worth knowing before assuming uniform cost
lets you just multiply one timing by `N_SHARDS`.

In [ ]:
N_SHARDS = 200
SAMPLE_KS = [1, 50, 100, 150, 200]

## Driver

One `--parallel k N_SHARDS --make-grm-bin` call per `k` in `SAMPLE_KS`, each
wrapped in `/usr/bin/time -v` for peak memory (`Maximum resident set size`)
alongside wall-clock -- measured, not estimated from the packed file size,
since plink's actual working set can be a multiple of that. Plink only --
no `GRM-pairs`/`grm_shard_tool` involved at this stage.

In [ ]:
import subprocess
import re

def get_n_ids(bed_prefix):
    with open(f"{bed_prefix}.fam") as f:
        return sum(1 for _ in f)

N_IDS = get_n_ids(BED_PREFIX)
print(f"N_IDS = {N_IDS}")

def time_shard(k, n_shards):
    out_prefix = os.path.join(LOCAL_WORK_DIR, f"grm_shard_{k}_of_{n_shards}")
    time_log = f"{out_prefix}.time.log"
    script = f'''
set -e
/usr/bin/time -v plink \
  --bfile "{BED_PREFIX}" \
  --make-grm-bin \
  --parallel {k} {n_shards} \
  --out "{out_prefix}" 2> "{time_log}"
'''
    result = subprocess.run(["bash", "-c", script], capture_output=True, text=True)

    elapsed_sec = None
    max_rss_kb = None
    if os.path.isfile(time_log):
        with open(time_log) as f:
            log_text = f.read()
        m = re.search(r"Elapsed \(wall clock\) time.*?: (.+)", log_text)
        if m:
            parts = [float(p) for p in re.split(r":", m.group(1).replace(",", "."))]
            elapsed_sec = parts[-1] + parts[-2] * 60 + (parts[-3] * 3600 if len(parts) > 2 else 0)
        m = re.search(r"Maximum resident set size \(kbytes\): (\d+)", log_text)
        if m:
            max_rss_kb = int(m.group(1))

    bin_size_bytes = os.path.getsize(f"{out_prefix}.grm.bin") if os.path.isfile(f"{out_prefix}.grm.bin") else None

    return {
        "k": k, "n_shards": n_shards,
        "elapsed_sec": elapsed_sec, "max_rss_gb": max_rss_kb / 1e6 if max_rss_kb else None,
        "bin_size_mb": bin_size_bytes / 1e6 if bin_size_bytes else None,
        "returncode": result.returncode, "time_log": time_log,
    }

shard_timings = [time_shard(k, N_SHARDS) for k in SAMPLE_KS]
for r in shard_timings:
    # elapsed_sec/max_rss_gb come back None if /usr/bin/time didn't produce the
    # expected output (e.g. it's not installed -- see Setup) -- format defensively
    # instead of crashing on a NoneType, so a partial failure is still readable
    status = "ok" if r["returncode"] == 0 else f"FAILED (see {r['time_log']})"
    elapsed = f"{r['elapsed_sec']:.1f}s" if r["elapsed_sec"] is not None else "N/A"
    rss = f"{r['max_rss_gb']:.1f} GB RSS" if r["max_rss_gb"] is not None else "N/A"
    print(f"k={r['k']:>3}: {elapsed}, {rss}, {status}")

## Feasibility summary

`elapsed_sec` across `SAMPLE_KS` should be roughly flat if plink's
row-balancing is working as documented -- a large spread means shard cost
isn't uniform and picking `N_SHARDS` needs more care than just dividing
total time by `n`. `max_rss_gb` is the real per-shard memory footprint,
measured, not estimated -- `416 GB ÷ max_rss_gb` is how many shards can
actually run concurrently on this machine, which is the number that
matters for wall-clock, not `N_CONCURRENT` guessed from vCPU count the way
the QC step did.

In [ ]:
import pandas as pd

timing_df = pd.DataFrame(shard_timings)
display(timing_df[["k", "elapsed_sec", "max_rss_gb", "bin_size_mb"]])

avg_shard_sec = timing_df["elapsed_sec"].mean()
spread_pct = 100 * (timing_df["elapsed_sec"].max() - timing_df["elapsed_sec"].min()) / avg_shard_sec
max_rss_gb = timing_df["max_rss_gb"].max()

MACHINE_RAM_GB = 416   # n1-highmem-64
RAM_SAFETY_FACTOR = 0.8   # leave headroom for the OS/other processes, not the full 416 GB

max_concurrent = int((MACHINE_RAM_GB * RAM_SAFETY_FACTOR) // max_rss_gb)

print(f"Average per-shard time: {avg_shard_sec:.1f} sec")
print(f"Spread across sampled k: {spread_pct:.1f}% (large -> row-balancing isn't as uniform as assumed)")
print(f"Peak memory per shard: {max_rss_gb:.1f} GB")
print(f"RAM-bounded max concurrent shards on this machine: {max_concurrent}")
print()

total_compute_hours = avg_shard_sec * N_SHARDS / 3600
print(f"Extrapolated total single-threaded compute for N_SHARDS={N_SHARDS}: {total_compute_hours:.1f} hours")

feasibility = pd.DataFrame({
    "n_concurrent": [1, max(1, max_concurrent // 2), max_concurrent],
}).assign(
    wall_clock_hours=lambda d: (N_SHARDS * avg_shard_sec / d["n_concurrent"]) / 3600
)
feasibility

## Next steps

If `spread_pct` is small and `wall_clock_hours` at `max_concurrent` looks
reasonable, `N_SHARDS=200` (or whatever was tested) is a reasonable choice
to commit to for the real sharded-GRM notebook -- run all `N_SHARDS` shards
at `max_concurrent` concurrency (same `ThreadPoolExecutor`-managed
`subprocess` pattern as `genome_wide_qc_thinning_merge.ipynb`'s driver,
capped by RAM this time instead of vCPUs). That notebook is plink only,
same as this one -- `GRM-pairs`/`grm_shard_tool` (`accumulate`/`merge`)
comes in later, as a separate step, for the phenotype cross-product against
the finished GRM shards (where GRM-side and phenotype-side `(FID, IID)`
alignment matters -- see `03_grm_shards/README.md`'s note on that). If
wall-clock still looks too long even at `max_concurrent`, the next lever is
`dsub`/Google Batch (separate machines, each with their own RAM budget)
rather than pushing concurrency higher on one machine.